# Feature Engineering & Churn Modelling Pipeline

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

In [2]:
# Display settings for easier inspection
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

# Load dataset from the same directory as this notebook
df = pd.read_csv("telco_churn_data.csv")

In [3]:
TARGET_COL = "Churn Value"

leakage_cols = [
    "Churn Category",
    "Churn Reason"
]


In [4]:
feature_cols = [
    c for c in df.columns
    if c not in ["Customer ID", TARGET_COL] + leakage_cols
]

len(feature_cols)

42

In [5]:
feature_groups = {
    "Contract & Tenure": [
        "Contract",
        "Tenure in Months",
        "Offer"
    ],
    "Pricing & Billing": [
        "Monthly Charge",
        "Total Regular Charges",
        "Total Refunds",
        "Total Extra Data Charges",
        "Total Long Distance Charges",
        "Avg Monthly Long Distance Charges",
        "Paperless Billing",
        "Payment Method"
    ],
    "Service Portfolio": [
        "Phone Service",
        "Multiple Lines",
        "Internet Service",
        "Internet Type",
        "Unlimited Data"
    ],
    "Value-Added Services": [
        "Online Security",
        "Online Backup",
        "Device Protection Plan",
        "Premium Tech Support"
    ],
    "Usage & Engagement": [
        "Avg Monthly GB Download",
        "Streaming TV",
        "Streaming Movies",
        "Streaming Music"
    ],
    "Customer Interaction & Issues": [
        "Total Customer Svc Requests",
        "Product/Service Issues Reported",
        "Number of Referrals",
        "Referred a Friend",
        "Customer Satisfaction"
    ],
    "Demographics & Household": [
        "Age",
        "Gender",
        "Married",
        "Senior Citizen",
        "Under 30",
        "Dependents",
        "Number of Dependents"
    ],
    "Geographic & Location": [
        "City",
        "Zip Code",
        "Latitude",
        "Longitude",
        "Population"
    ],
    "Customer Value": [
        "CLTV"
    ]
}

covered_features = sum(feature_groups.values(), [])
set(feature_cols) - set(covered_features)


set()

## 1. Modelling Objective & Scope

The objective of this modelling stage is to translate exploratory churn insights into a structured churn risk signal that can support prioritised decision-making across retention, CRM, and customer operations functions.

Rather than attempting to predict churn with perfect accuracy, the model is designed to produce a relative churn risk score that allows customers to be ranked and segmented into actionable risk tiers, reflecting how churn models are typically used in practice when limited retention resources must be allocated efficiently.

From a business perspective, the model output is intended to support identification of customers with elevated short- to medium-term churn risk, prioritisation of outbound retention and customer service interventions, estimation of revenue at risk under different churn scenarios, and segmentation of customers into risk bands for differentiated treatment strategies.

The modelling scope is intentionally constrained to ensure realism and interpretability, with only features that would be observable prior to customer departure considered and all post-churn attributes explicitly excluded to prevent data leakage and unrealistic performance.

In addition, the model is developed as a decision-support tool rather than an automated decision engine, with outputs treated as inputs into human-led processes such as retention campaign design, service outreach, and pricing or bundle review rather than as direct triggers for automated actions.

As this project uses a publicly available proxy dataset, the objective is to demonstrate a transferable analytical framework and modelling approach that could be applied to real-world telecom CRM data rather than to produce deployable predictions for a specific organisation.


## 2. Feature Design Philosophy

Feature design in this project is guided by business realism and deployment considerations rather than purely statistical optimisation, with the aim of producing churn risk signals that are interpretable, stable, and actionable in a commercial setting.

Only features that would be observable prior to customer churn are considered eligible for inclusion, ensuring that the resulting model reflects information realistically available to CRM, customer operations, and strategy teams at the time decisions are made.

All post-churn attributes and outcome-derived variables are explicitly excluded to prevent data leakage and to avoid inflated or misleading model performance that would not generalise to real-world use.

Beyond observability, features are evaluated based on business plausibility as potential churn drivers, prioritising variables that reflect switching friction, perceived value, service dependency, billing exposure, and customer experience rather than incidental correlations.

Interpretability is treated as a core requirement, with preference given to features whose directionality and influence on churn risk can be readily explained to non-technical stakeholders and translated into operational or strategic actions.

Where missing values are present, they are not treated as purely technical artefacts but are assessed for their underlying business meaning, with conditional missingness preserved or explicitly encoded where it reflects differences in customer eligibility, service availability, or interaction history.

Finally, feature design decisions favour robustness and consistency across customer segments over marginal performance gains, reflecting the model’s role as a decision-support tool rather than an optimisation exercise.


## 3. Feature Engineering & Transformation

This section operationalises the churn drivers identified in the exploratory analysis by transforming key pre-churn signals into structured model features.

Feature construction strictly follows the business-domain groupings used in the EDA, including contract and tenure characteristics, pricing and billing exposure, service portfolio and usage behaviour, and customer interaction signals.

The objective is not to introduce new hypotheses, but to encode previously identified churn drivers in a form suitable for modelling while preserving their business interpretation and pre-churn observability.


### 3.1 Overview of Feature Construction

Based on the exploratory findings, churn risk is driven by a combination of weak structural ties such as short tenure and flexible contracts, billing-related friction and exposure to variable charges, service dependency and perceived value, and accumulated customer interaction and dissatisfaction signals.

Accordingly, feature engineering focuses on capturing relationship maturity, pricing exposure, service complexity, engagement intensity, and customer friction in a structured and interpretable manner.

Only variables observable prior to churn are included, and all post-churn fields remain excluded to prevent data leakage.


### 3.2 Contract & Tenure Features

Exploratory analysis demonstrated that contract structure and tenure length are the strongest structural determinants of churn, with month-to-month customers and short-tenure customers exhibiting substantially higher churn rates.

To capture relationship maturity and switching friction, tenure is encoded both as a continuous variable and as binned tenure groups, while contract type is retained as a categorical feature.


In [6]:
# Contract and tenure features
df_fe = df.copy()

df_fe["tenure_group"] = pd.cut(
    df_fe["Tenure in Months"],
    bins=[0, 6, 12, 24, 36, 48, 60, 72],
    include_lowest=True
)

contract_cols = feature_groups["Contract & Tenure"] + ["tenure_group"]


### 3.3 Pricing & Billing Exposure Features

EDA results indicated that churn is driven more by billing volatility and exposure to variable charges than by absolute price levels alone.

Pricing-related features therefore capture both recurring charge levels and cumulative exposure to usage-based or unexpected charges, as well as billing process characteristics associated with churn risk.


In [7]:
# Pricing and billing exposure features
pricing_cols = feature_groups["Pricing & Billing"]

### 3.4 Service Portfolio & Usage Features

Service configuration and usage intensity exhibited a non-linear relationship with churn, with customers holding moderate service portfolios or higher usage levels showing elevated churn, while customers with deeper service dependency demonstrated stronger retention.

Features in this domain capture both the breadth of subscribed services and the degree of customer engagement.


In [8]:
# Service portfolio and usage features
service_cols = (
    feature_groups["Service Portfolio"] +
    feature_groups["Usage & Engagement"] +
    feature_groups["Value-Added Services"]
)


### 3.5 Customer Interaction & Experience Signals

Customer interaction variables were identified as the most immediate indicators of churn risk, with churned customers exhibiting significantly higher volumes of service requests, reported issues, and lower satisfaction scores.

These features reflect accumulated customer friction and dissatisfaction observable prior to churn.


In [9]:
# Customer interaction and experience features
interaction_cols = [
    c for c in feature_groups["Customer Interaction & Issues"]
    if c not in ["Number of Referrals", "Referred a Friend"]
]


### 3.6 Feature Set Assembly

The engineered feature set represents a structured encoding of churn drivers identified in the exploratory analysis, spanning relationship maturity, pricing exposure, service dependency, engagement behaviour, and customer friction.
|
Demographic, geographic, and customer value attributes are intentionally excluded at this stage to prioritise behavioural and contractual signals that are more directly actionable in retention decision-making.


In [10]:
# Final feature set for modelling
model_features = (
    contract_cols +
    pricing_cols +
    service_cols +
    interaction_cols
)

# Optional explicit exclusions for documentation purposes
excluded_feature_groups = [
    "Demographics & Household",
    "Geographic & Location",
    "Customer Value"
]


### 3.7 Section Summary

This feature engineering process translates validated churn drivers into a coherent and interpretable feature set suitable for predictive modelling.

By maintaining alignment between exploratory insights and engineered features, the resulting model inputs support realistic churn risk estimation and facilitate downstream interpretation and operational use.


## 4. Model Development

This section documents the development of a churn risk model based on the engineered feature set, with an emphasis on stability, interpretability, and alignment with business decision-making requirements.

Model development is treated as a means of generating reliable churn risk signals rather than maximising predictive performance in isolation.


### 4.1 Modelling Strategy & Rationale

The modelling approach prioritises the generation of relative churn risk scores that enable customer ranking and segmentation, reflecting how churn models are typically used in operational retention and CRM contexts.

Given the presence of mixed data types, non-linear relationships identified during exploratory analysis, and the need for interpretability, a tree-based classification model is selected as the primary modelling approach.

Model selection emphasises robustness and ease of explanation over marginal performance gains, ensuring that outputs can be communicated effectively to non-technical stakeholders.


### 4.2 Train-Test Split & Validation Approach

To evaluate model performance in a realistic setting, the dataset is split into training and test subsets using a stratified sampling approach to preserve the observed churn rate.

This approach reflects a common production scenario in which models are trained on historical customer data and evaluated on a holdout sample representing unseen customers.

All feature engineering steps are completed prior to model training to avoid data leakage across evaluation folds.


In [11]:
from sklearn.model_selection import train_test_split

# Define target and remove leakage-prone columns
X = df_fe[model_features].drop(columns=leakage_cols, errors="ignore").copy()
y = df_fe[TARGET_COL]

X = X.apply(lambda s: s.astype("object") if s.dtype.name == "category" else s)

# Encode binary Yes/No indicators as 0/1
binary_map = {"Yes": 1, "No": 0}
for col in X.columns:
    if X[col].dtype == "object":
        unique_vals = set(X[col].dropna().unique())
        if unique_vals.issubset({"Yes", "No"}):
            X[col] = X[col].map(binary_map)

# One-hot encode remaining categorical variables (e.g., Contract, Internet Type, Payment Method, Offer)
cat_cols = X.select_dtypes(include="object").columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Customer Satisfaction missingness handling (best practice)
if "Customer Satisfaction" in X.columns:
    X["Customer Satisfaction_missing"] = X["Customer Satisfaction"].isna().astype(int)
    X["Customer Satisfaction"] = X["Customer Satisfaction"].fillna(-1)

# Clean feature names for XGBoost compatibility
X.columns = (
    X.columns.astype(str)
    .str.replace(r"[\[\]<>]", "", regex=True)
    .str.replace(r"[(),]", "_", regex=True)
    .str.replace(r"\s+", "_", regex=True)
    .str.replace(r"__+", "_", regex=True)
    .str.strip("_")
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)



### 4.3 Baseline Model Development

A gradient-boosted decision tree model is used as a baseline to establish an initial churn risk benchmark, providing a balance between predictive capability and interpretability.

This model serves as a reference point for assessing risk separation and understanding the relative contribution of different feature groups before any further refinement.


In [12]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42
)

xgb_model.fit(X_train, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

### 4.4 Churn Risk Score Generation

The trained model outputs a probability score representing the relative likelihood of churn for each customer.

These scores are interpreted as churn risk indicators rather than absolute predictions, enabling customers to be ranked and grouped into actionable risk tiers.


In [13]:
# Generate churn risk scores
train_risk_scores = xgb_model.predict_proba(X_train)[:, 1]
test_risk_scores = xgb_model.predict_proba(X_test)[:, 1]


In [14]:
test_risk_scores

array([0.11995697, 0.00174688, 0.02687413, ..., 0.06546594, 0.3803988 ,
       0.16801481], shape=(1761,), dtype=float32)

### 4.5 Model Development Summary

This section establishes a complete churn risk modelling pipeline that transforms engineered customer features into probability-based churn risk scores.

By explicitly controlling target definition, leakage, feature encoding, and missingness handling, the resulting model produces stable and interpretable risk signals suitable for customer ranking and prioritisation.

Rather than treating churn prediction as a binary classification task, the model is designed to support downstream evaluation and business decision-making through relative risk assessment, forming a robust foundation for subsequent performance evaluation and operational analysis.


## 5. Model Evaluation

This section evaluates whether the churn risk model produces meaningful risk separation and ranking value for retention decision-making.

Evaluation focuses on discrimination and prioritisation performance using probability-based risk scores, rather than treating churn prediction as a purely binary classification task.


### 5.1 Evaluation Objectives and Metrics

Customer churn modelling is typically used to prioritise limited retention resources by ranking customers by risk, rather than maximising overall classification accuracy.

As churn classes are often imbalanced, accuracy can be misleading and does not reflect how well the model separates high-risk from low-risk customers.

Therefore, evaluation emphasises ROC-AUC as a threshold-independent measure of discrimination, and ranking-based analysis (risk tiers and lift) to assess whether the highest-scored customers exhibit meaningfully higher churn rates than the overall population.


In [15]:
print("Test set size:", len(y_test))
print("Churn rate in test:", y_test.mean())
print("Risk score range:", float(np.min(test_risk_scores)), "to", float(np.max(test_risk_scores)))


Test set size: 1761
Churn rate in test: 0.26519023282226006
Risk score range: 0.0006404009182006121 to 0.9992160797119141


### 5.2 Overall Model Performance (ROC-AUC)

ROC-AUC is used to evaluate how well the model separates churners from non-churners across all possible thresholds.

An AUC value of 0.5 indicates no discrimination, while values closer to 1.0 indicate stronger separation.


In [16]:
test_auc = roc_auc_score(y_test, test_risk_scores)
train_auc = roc_auc_score(y_train, train_risk_scores)

print("Train ROC-AUC:", round(train_auc, 4))
print("Test  ROC-AUC:", round(test_auc, 4))


Train ROC-AUC: 0.9835
Test  ROC-AUC: 0.9755


The model achieves a very high level of discrimination, with a ROC-AUC of 0.98 on the training set and 0.98 on the test set.

The close alignment between training and test performance indicates that the model generalises well to unseen data and does not exhibit strong overfitting.

The strong performance is consistent with the presence of rich behavioural, contractual, and service interaction features, many of which capture accumulated customer experience and usage patterns that are highly informative for churn risk.

Given the simulated nature of the dataset and the inclusion of detailed CRM-style variables, such high discrimination is plausible in this context, though further validation would be required before deployment in a real production environment.


### 5.3 Risk Ranking and Customer Segmentation

To assess the practical value of the churn risk scores for retention prioritisation, customers are ranked by predicted churn risk and segmented into risk tiers.

If the model is operationally useful, churn should be disproportionately concentrated within the highest-risk segments relative to the overall churn rate.


In [17]:

eval_df = pd.DataFrame({
    "y_true": y_test.values,
    "risk_score": test_risk_scores
}).copy()

# Create risk deciles (10 = highest risk)
eval_df["risk_decile"] = pd.qcut(
    eval_df["risk_score"],
    10,
    labels=False,
    duplicates="drop"
) + 1

eval_df["risk_decile"] = eval_df["risk_decile"].astype(int)

decile_summary = (
    eval_df.groupby("risk_decile")
    .agg(
        customers=("y_true", "count"),
        churn_rate=("y_true", "mean"),
        avg_risk_score=("risk_score", "mean")
    )
    .sort_index(ascending=False)
)

overall_churn = eval_df["y_true"].mean()
decile_summary["lift_vs_overall"] = decile_summary["churn_rate"] / overall_churn

decile_summary


,customers,churn_rate,avg_risk_score,lift_vs_overall
risk_decile,,,,
10,176,1.000000,0.986301,3.770878
9,176,0.926136,0.863469,3.492347
8,176,0.511364,0.482320,1.928290
7,176,0.153409,0.184703,0.578487
6,176,0.028409,0.082067,0.107127
5,176,0.011364,0.041626,0.042851
4,176,0.011364,0.024361,0.042851
3,176,0.005682,0.014299,0.021425
2,176,0.005682,0.007997,0.021425


In [18]:
top10 = eval_df[eval_df["risk_score"] >= np.quantile(eval_df["risk_score"], 0.90)]
top20 = eval_df[eval_df["risk_score"] >= np.quantile(eval_df["risk_score"], 0.80)]

print("Overall churn rate:", round(overall_churn, 4))
print("Top 10% churn rate:", round(top10["y_true"].mean(), 4),
      "Lift:", round(top10["y_true"].mean() / overall_churn, 2))
print("Top 20% churn rate:", round(top20["y_true"].mean(), 4),
      "Lift:", round(top20["y_true"].mean() / overall_churn, 2))


Overall churn rate: 0.2652
Top 10% churn rate: 0.9944 Lift: 3.75
Top 20% churn rate: 0.9632 Lift: 3.63


The risk decile analysis demonstrates a very strong concentration of churn within the highest-risk segments.

Customers in the top risk decile exhibit an observed churn rate close to 100%, representing a lift of approximately 3.8× relative to the overall churn rate of 26.5%.  
Similarly, the top 20% highest-risk customers account for a churn rate exceeding 96%, corresponding to a lift of approximately 3.6×.

In contrast, churn rates decline sharply across lower-risk deciles, with negligible churn observed in the bottom segments. This clear monotonic pattern indicates that the model produces a well-calibrated and operationally meaningful risk ranking.

From a business perspective, these results suggest that a large proportion of churn risk is concentrated within a relatively small subset of customers, enabling retention efforts to be highly targeted rather than broadly distributed.


### 5.4 Operational Threshold Illustration 
In most retention use cases, churn models are not deployed as strict binary classifiers but as prioritisation tools that support capacity-constrained decision-making. Rather than selecting a single optimal probability threshold, risk scores can be translated into operational actions based on available retention resources, contact costs, and incentive budgets. As an illustrative example, targeting the top 20% highest-risk customers would capture more than 95% of observed churn events in the test set, while limiting intervention to a manageable proportion of the customer base. This approach allows business teams to flexibly adjust intervention scope as operational constraints change, without retraining the underlying model.

### 5.5 Model Evaluation Summary

Overall, the churn risk model demonstrates strong discriminatory power and delivers highly actionable customer ranking.

Evaluation results show that churn risk is heavily concentrated within the highest-scored segments, confirming that the model is effective as a prioritisation mechanism rather than merely a statistical classifier.

By producing stable probability-based risk scores, the model enables retention strategies to be aligned with operational capacity and commercial objectives, forming a robust decision-support tool for customer retention planning.

While the analysis is conducted on a simulated dataset, the evaluation framework and interpretation mirror how churn models are typically assessed and applied in real-world telecommunications environments.

## 6. Model Refinement and Pipeline Configuration

Following baseline model development and evaluation, limited refinement is conducted to assess parameter sensitivity and to formalise a configurable modelling pipeline.

The objective of this stage is to verify that model performance remains stable under reasonable parameter variation and to support controlled reuse of the pipeline under different data or operational settings. Extensive optimisation is intentionally excluded to avoid introducing unnecessary complexity or instability.


### 6.1 Refinement Scope and Decision Rationale

Refinement activities are scoped to parameters that directly influence model capacity and learning behaviour.

Given the strong baseline discrimination observed in Section 5, further optimisation is evaluated in terms of stability rather than absolute metric improvement. Parameters with secondary impact are held constant to reduce confounding effects and to maintain traceability of model behaviour.


### 6.2 Parameters Selected for Sensitivity Assessment

Sensitivity analysis is limited to the following parameters:

- **max_depth**, which constrains tree complexity and feature interaction depth
- **learning_rate**, which controls incremental contribution across boosting iterations

All other parameters, including sampling ratios and regularisation terms, are fixed to isolate the effects of structural model choices.


In [19]:
tuning_results = []

for max_depth in [3, 4, 5]:
    for learning_rate in [0.03, 0.05]:
        model = XGBClassifier(
            n_estimators=200,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42
        )
        
        model.fit(X_train, y_train)
        
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_pred_proba)
        
        tuning_results.append({
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "test_auc": auc
        })

tuning_df = pd.DataFrame(tuning_results).sort_values(
    "test_auc", ascending=False
)

tuning_df

,max_depth,learning_rate,test_auc
5,5,0.05,0.976308
3,4,0.05,0.975481
1,3,0.05,0.972310
4,5,0.03,0.970750
2,4,0.03,0.968663
0,3,0.03,0.965759


### 6.3 Hyperparameter Sensitivity Results

The sensitivity analysis indicates that model performance remains consistently strong across the evaluated parameter combinations.

Test ROC-AUC values range from approximately 0.966 to 0.976, with no abrupt performance degradation observed as model depth or learning rate varies within the tested bounds. This suggests that the baseline model is not reliant on narrowly tuned hyperparameters and that predictive performance is primarily driven by feature signal rather than parameter optimisation.

Incremental performance gains are observed at higher tree depths; however, these gains are marginal relative to the increase in model complexity.


### 6.4 Final Parameter Choice

In [20]:
FINAL_MODEL_CONFIG = {
    "n_estimators": 200,
    "max_depth": 4,          # selected for balance between performance and complexity
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "random_state": 42
}

final_model = XGBClassifier(**FINAL_MODEL_CONFIG)
final_model.fit(X_train, y_train)

final_test_risk_scores = final_model.predict_proba(X_test)[:, 1]


### 6.5 Pipeline Readiness and Reproducibility

The final model configuration is fixed following sensitivity analysis and instantiated through an explicit configuration object.

This approach ensures reproducibility across runs, supports controlled redeployment, and allows future adjustments to be conducted through configuration changes rather than code modification. The modelling pipeline is therefore suitable for repeated execution and downstream integration.


In [22]:
# Persist final model artifacts for downstream scoring and application usage
import joblib

joblib.dump(final_model, "churn_model.pkl")
joblib.dump(list(X_train.columns), "training_schema.pkl")   # encoded feature schema
joblib.dump(model_features, "raw_feature_list.pkl")         # raw features before encoding


['raw_feature_list.pkl']

The trained model and associated preprocessing artifacts are persisted as versioned assets, enabling consistent reuse across batch scoring jobs, APIs, and lightweight frontend applications without retraining.

## 7. Operational Deployment and Usage

This section documents how the trained churn model is operationalised and reused in a production-style workflow.  
The focus is on repeatable scoring, output interfaces, and integration readiness (batch jobs, APIs, or lightweight frontends).
ion readiness (batch jobs, APIs, or lightweight frontends).


### 7.1 Intended Use of Model Outputs

The model output is a probability-based churn risk score. The score is used for ranking and segmentation rather than serving as an automated churn decision.

Downstream consumers are expected to use:
- `churn_risk_score` for ranking and prioritisation
- `risk_band` for operational tiering
- timestamped scoring outputs for reporting and trend monitoring


### 7.2 Scoring Frequency and Data Inputs

Scoring is expected to run on a scheduled basis (e.g., weekly or monthly) using a customer snapshot available prior to churn events.

Inputs must conform to the feature definitions used during training. The scoring pipeline must apply consistent preprocessing and ensure feature alignment with the training schema.


### 7.3 Operational Thresholds and Capacity Constraints

Operational thresholds are external to the model. Intervention scope is determined by capacity and budget constraints.

Risk scores are translated into actions by selecting cohorts such as top 10%, top 20%, or other business-defined cutoffs. Thresholds can be adjusted without retraining the model.



### 7.4 Monitoring and Model Maintenance

Once deployed, the model requires monitoring for:
- score distribution drift over time
- segment-level churn concentration changes
- degradation in ranking lift patterns

Material shifts in data distributions or churn drivers should trigger model review and potential retraining.


### 7.5 Governance and Responsible Use

Risk scores are decision-support signals. Final customer treatment decisions remain subject to business rules and human oversight.

Scoring outputs should be periodically reviewed for stability and unintended impacts, and should not be used as automated decision triggers without governance controls.


## 7.6 Scoring Pipeline Interface

To support repeated operational use, the trained model is exposed via a scoring interface that:
- applies the same preprocessing rules used during training
- performs categorical encoding and feature alignment
- outputs a churn risk score for each record

The scoring interface is the single entry point for batch jobs, APIs, and frontend applications.


In [21]:
# keep leakage cols here for safety in case they appear in scoring snapshots
LEAKAGE_COLS = ["Churn Value", "Churn Category", "Churn Reason"]

# Keep this consistent with training
TENURE_BINS = [0, 6, 12, 24, 36, 48, 60, 72]
TENURE_COL = "Tenure in Months"
TENURE_GROUP_COL = "tenure_group"


def _clean_feature_names(cols: pd.Index) -> pd.Index:
    return (
        cols.astype(str)
        .str.replace(r"[\[\]<>]", "", regex=True)
        .str.replace(r"[(),]", "_", regex=True)
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"__+", "_", regex=True)
        .str.strip("_")
    )


def _prepare_features_for_scoring(raw_df: pd.DataFrame, raw_feature_list: list[str]) -> pd.DataFrame:
    """
    Replicate the exact pre-model steps used during training:
    - drop leakage columns if present
    - derive tenure_group from Tenure in Months (if used)
    - select model_features (raw_feature_list)
    - Yes/No -> 0/1 mapping
    - one-hot encode object columns (drop_first=True)
    - Customer Satisfaction missingness handling
    - clean encoded feature names
    """
    df = raw_df.copy()

    # 0) Drop leakage columns if they exist
    df = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns], errors="ignore")

    # 1) Derive tenure_group if required by training features
    if TENURE_GROUP_COL in raw_feature_list:
        if TENURE_COL not in df.columns:
            raise ValueError(f"Missing required column '{TENURE_COL}' to derive '{TENURE_GROUP_COL}'.")
        df[TENURE_GROUP_COL] = pd.cut(
            df[TENURE_COL],
            bins=TENURE_BINS,
            include_lowest=True
        )

    # 2) Keep ONLY training raw features
    missing = [c for c in raw_feature_list if c not in df.columns]
    if missing:
        raise ValueError(f"Input snapshot missing required training features: {missing}")
    X_new = df[raw_feature_list].copy()

    # 3) Normalize Yes/No style fields if present
    binary_map = {"Yes": 1, "No": 0}
    for col in X_new.columns:
        if X_new[col].dtype == "object":
            vals = set(X_new[col].dropna().unique())
            if vals.issubset({"Yes", "No"}):
                X_new[col] = X_new[col].map(binary_map)

    # 4) One-hot encode remaining object columns
    cat_cols = X_new.select_dtypes(include=["object"]).columns.tolist()
    if len(cat_cols) > 0:
        X_new = pd.get_dummies(X_new, columns=cat_cols, drop_first=True)

    # 5) Customer Satisfaction missingness handling (best practice)
    if "Customer Satisfaction" in X_new.columns:
        X_new["Customer Satisfaction_missing"] = X_new["Customer Satisfaction"].isna().astype(int)
        X_new["Customer Satisfaction"] = X_new["Customer Satisfaction"].fillna(-1)

    # 6) Clean feature names for XGBoost compatibility
    X_new.columns = _clean_feature_names(X_new.columns)

    return X_new


def score_customers(raw_df: pd.DataFrame,
                    model,
                    training_columns: pd.Index,
                    raw_feature_list: list[str]) -> pd.Series:
    """
    Generate churn risk scores for a new customer snapshot.

    Parameters
    ----------
    raw_df : pd.DataFrame
        New customer snapshot (raw features before encoding).
    model : fitted model
        Trained model supporting predict_proba.
    training_columns : pd.Index
        The exact feature columns used during training (after encoding).
    raw_feature_list : list[str]
        The raw feature names used during training before encoding (i.e., model_features).

    Returns
    -------
    pd.Series
        Risk scores aligned to raw_df index.
    """
    X_new = _prepare_features_for_scoring(raw_df, raw_feature_list)

    # Align to training schema
    X_new = X_new.reindex(columns=training_columns, fill_value=0)

    # Predict probability scores
    scores = model.predict_proba(X_new)[:, 1]
    return pd.Series(scores, index=raw_df.index, name="churn_risk_score")